# 14 · Final Test Evaluation and Temporal Check

**What this notebook covers**

- Integrity checks on the locked policy, model and gate report
- Score the Test split once, with bootstrap confidence intervals
- Dev vs Test gap, confusion matrix and risk tiers
- Expected business cost vs no model
- Appendix: train on 2015 to 2016, test on 2017

## 1. Environment setup
### 1.1 Imports

In [ ]:
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.base import clone
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

### 1.2 Configuration

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_PREFIX = "14"
N_BOOTSTRAP = 2000
ALLOW_RESCORE = False          # True only to reproduce the SAME locked evaluation (identical policy hash)
RUN_TEMPORAL_CHECK = True
TEMPORAL_TRAIN_YEARS = [2015, 2016]
TEMPORAL_TEST_YEARS = [2017]

### 1.3 Project paths, shared helpers and plotting style

In [ ]:
# Resolve the repository root regardless of where the kernel was started
# (VS Code starts kernels in notebooks/, Jupyter may start them at the repo root).
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "DECISIONS.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Project root not found (no DECISIONS.md in any parent directory).")


PROJECT_ROOT = find_project_root()

PATHS = {
    "raw": PROJECT_ROOT / "data" / "hotel_bookings.csv",
    "clean": PROJECT_ROOT / "data" / "interim" / "hotel_bookings_clean.csv",
    "splits": PROJECT_ROOT / "data" / "processed" / "splits",
    "features": PROJECT_ROOT / "data" / "processed" / "features",
    "artifacts": PROJECT_ROOT / "artifacts",
    "preprocessors": PROJECT_ROOT / "artifacts" / "preprocessors",
    "model_outputs": PROJECT_ROOT / "artifacts" / "model_outputs",
    "models": PROJECT_ROOT / "models",
    "figures": PROJECT_ROOT / "reports" / "figures",
    "tables": PROJECT_ROOT / "reports" / "tables",
    "notebooks": PROJECT_ROOT / "notebooks",
}
for key in ("artifacts", "preprocessors", "model_outputs", "models", "figures", "tables"):
    PATHS[key].mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Columns that must always be treated as categorical strings when re-reading CSVs.
STRING_COLUMNS = [
    "hotel", "arrival_date_month", "meal", "country", "market_segment", "distribution_channel",
    "reserved_room_type", "assigned_room_type", "deposit_type", "agent", "company",
    "customer_type", "arrival_season",
]
TARGET = "is_canceled"
ID_COLUMN = "booking_id"
LEAKAGE_COLUMNS = ["reservation_status", "reservation_status_date"]

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Streaming SHA-256 of a file (used to fingerprint every artifact)."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact missing: {path.relative_to(PROJECT_ROOT)}")
    return json.loads(path.read_text())


def read_frame(path: Path) -> pd.DataFrame:
    """Read a project CSV with categorical columns forced to plain Python strings.

    `keep_default_na=False` + an explicit NA list stops pandas from turning legitimate
    category labels into NaN; numeric columns are parsed normally.
    """
    if not path.exists():
        raise FileNotFoundError(f"Required data file missing: {path.relative_to(PROJECT_ROOT)}")
    frame = pd.read_csv(
        path,
        dtype={col: object for col in STRING_COLUMNS},
        keep_default_na=False,
        na_values=["", "NaN", "nan"],
    )
    present = [c for c in STRING_COLUMNS if c in frame.columns]
    frame[present] = frame[present].astype(object)
    return frame

In [ ]:
# One visual language for every figure in the project:
#  - blue = not cancelled, orange = cancelled (validated colour-blind-safe pair)
#  - single-series charts use blue; recessive grey grid; no top/right spines; no dual axes.
COLORS = {
    "not_canceled": "#2a78d6",
    "canceled": "#eb6834",
    "neutral": "#8a8985",
    "ink": "#0b0b0b",
    "ink_muted": "#52514e",
}
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQUENTIAL_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "project_blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
DIVERGING_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "project_diverging", ["#2a78d6", "#e8e7e3", "#eb6834"]
)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "axes.prop_cycle": mpl.cycler(color=SERIES_COLORS),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae",
    "axes.labelcolor": COLORS["ink_muted"],
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.titlecolor": COLORS["ink"],
    "axes.grid": True,
    "grid.color": "#e8e7e3",
    "grid.linewidth": 0.8,
    "xtick.color": COLORS["ink_muted"],
    "ytick.color": COLORS["ink_muted"],
    "lines.linewidth": 2,
    "legend.frameon": False,
})


def save_figure(fig, name: str) -> Path:
    """Save a figure under reports/figures with this notebook's numeric prefix."""
    path = PATHS["figures"] / f"{NOTEBOOK_PREFIX}_{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"Saved figure -> {path.relative_to(PROJECT_ROOT)}")
    return path

In [ ]:
import importlib.metadata as importlib_metadata

def library_versions(packages=("numpy", "pandas", "scikit-learn", "xgboost", "matplotlib", "seaborn", "scipy", "shap", "joblib")) -> dict:
    versions = {"python": sys.version.split()[0], "platform": platform.platform()}
    for package in packages:
        try:
            versions[package] = importlib_metadata.version(package)
        except importlib_metadata.PackageNotFoundError:
            versions[package] = "not installed"
    return versions

LIBRARY_VERSIONS = library_versions()
pd.Series(LIBRARY_VERSIONS, name="version").to_frame()

### 1.4 Calibration helpers (same definitions as notebook 12)

In [ ]:
EPS = 1e-6


def fit_calibrator(method: str, p_raw, y):
    """Fit a probability-level calibrator. 'none' returns None (identity)."""
    p_raw = np.clip(np.asarray(p_raw, dtype=float), EPS, 1 - EPS)
    if method == "none":
        return None
    if method == "sigmoid":          # Platt scaling on the logit of the raw score
        return LogisticRegression(C=1e6, max_iter=1000).fit(logit(p_raw).reshape(-1, 1), y)
    if method == "isotonic":
        return IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(p_raw, y)
    raise ValueError(method)


def apply_calibration(method: str, calibrator, p_raw):
    p_raw = np.clip(np.asarray(p_raw, dtype=float), EPS, 1 - EPS)
    if method == "none":
        return p_raw
    if method == "sigmoid":
        return calibrator.predict_proba(logit(p_raw).reshape(-1, 1))[:, 1]
    return np.clip(calibrator.predict(p_raw), 0.0, 1.0)

### 1.5 The unified metric function (same definition as notebook 11)

In [ ]:
def evaluate_predictions(y_true, y_proba, threshold: float) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.clip(np.asarray(y_proba, dtype=float), 1e-7, 1 - 1e-7)
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "pr_auc": average_precision_score(y_true, y_proba),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "brier": brier_score_loss(y_true, y_proba),
        "log_loss": log_loss(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "flag_rate": y_pred.mean(),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "threshold": threshold,
    }

## 2. Pre-flight integrity checks
### 2.1 Policy, bundle and gate must be exactly as locked
The policy hash is recomputed from its contents. Any edit to the threshold, tiers or
model after notebook 12 makes the hash mismatch and stops the notebook.

In [ ]:
policy = load_json(PATHS["artifacts"] / "decision_policy.json")
policy_core = {k: v for k, v in policy.items() if k not in {"policy_sha256", "created_at", "notebook"}}
recomputed = hashlib.sha256(json.dumps(policy_core, sort_keys=True, default=str).encode()).hexdigest()
assert recomputed == policy["policy_sha256"], "Decision policy was edited after it was locked in notebook 12."
assert policy["selected_on"] == "dev", "Threshold/calibration must have been selected on Dev."

bundle_path = PROJECT_ROOT / policy["bundle_path"]
assert sha256_file(bundle_path) == policy["bundle_sha256"], "Final model bundle changed after the policy was locked."

gate = load_json(PATHS["artifacts"] / "validation_gate_report.json")
assert gate["global_passed"] and policy["model_id"] in gate["eligible_models"], "Selected model is not gate-eligible."
print(f"Policy {policy['policy_sha256'][:16]}... verified | model {policy['model_id']} | threshold {policy['operating_threshold']:.2f}")

### 2.2 Test-once lock
If a lock already exists, the locked results are shown and the notebook stops scoring.

In [ ]:
LOCK_PATH = PATHS["artifacts"] / "test_evaluation_lock.json"
ALREADY_LOCKED = LOCK_PATH.exists()
if ALREADY_LOCKED:
    lock = load_json(LOCK_PATH)
    assert lock["policy_sha256"] == policy["policy_sha256"], (
        "Test was already evaluated under a DIFFERENT policy. Re-scoring would leak Test into model development."
    )
    print(f"Test already evaluated at {lock['evaluated_at']} under this policy.")
    display(pd.Series(lock["headline_metrics"], name="locked test metrics").to_frame())
    if not ALLOW_RESCORE:
        print("ALLOW_RESCORE is False -> showing locked results only. Sections 3-5 will reuse them.")
SCORE_TEST = (not ALREADY_LOCKED) or ALLOW_RESCORE
print(f"Score Test in this run: {SCORE_TEST}")

## 3. Score the Test split (once)
### 3.1 Load Test features and verify the fingerprint

In [ ]:
feature_spec = load_json(PATHS["artifacts"] / "feature_spec.json")
TEST_PATH = PATHS["features"] / "test.csv"
test_hash = sha256_file(TEST_PATH)
assert test_hash == feature_spec["file_hashes"]["test"], "Test feature file changed - re-run notebook 04."

if SCORE_TEST:
    test = read_frame(TEST_PATH)
    X_test, y_test = test.drop(columns=[TARGET, ID_COLUMN]), test[TARGET].astype(int).to_numpy()
    print(f"Test: {X_test.shape}, prevalence {y_test.mean():.4f}")

### 3.2 Predict with the locked bundle and compute metrics

In [ ]:
bundle = joblib.load(bundle_path)
THRESHOLD = policy["operating_threshold"]
MEDIUM = policy["tiers"]["Medium"][0]

if SCORE_TEST:
    p_raw_test = bundle["model"].predict_proba(X_test)[:, 1]
    p_test = apply_calibration(bundle["calibration_method"], bundle["calibrator"], p_raw_test)
    test_metrics = evaluate_predictions(y_test, p_test, THRESHOLD)
    pd.DataFrame({ID_COLUMN: test[ID_COLUMN], "y_true": y_test, "p_raw": p_raw_test, "p_calibrated": p_test}).to_csv(
        PATHS["tables"] / "14_test_predictions.csv", index=False)
else:
    saved = pd.read_csv(PATHS["tables"] / "14_test_predictions.csv")
    y_test, p_test = saved["y_true"].to_numpy(), saved["p_calibrated"].to_numpy()
    test_metrics = evaluate_predictions(y_test, p_test, THRESHOLD)
pd.Series(test_metrics).to_frame("test")

### 3.3 Bootstrap 95 % confidence intervals on Test
Single numbers hide sampling uncertainty. The report should quote these intervals.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
boot = {m: [] for m in ["pr_auc", "roc_auc", "precision", "recall", "f1", "brier"]}
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, len(y_test), size=len(y_test))
    m = evaluate_predictions(y_test[idx], p_test[idx], THRESHOLD)
    for key in boot:
        boot[key].append(m[key])
ci_table = pd.DataFrame({
    "estimate": {k: test_metrics[k] for k in boot},
    "ci_low": {k: np.quantile(v, 0.025) for k, v in boot.items()},
    "ci_high": {k: np.quantile(v, 0.975) for k, v in boot.items()},
})
ci_table.round(4)

### 3.4 Dev → Test generalisation gap
Dev numbers are out-of-fold (notebook 12). A small gap means the selection and
threshold process did not over-fit Dev. A large drop must be reported and explained.

In [ ]:
dev_oof = policy["dev_out_of_fold"]
gap = pd.DataFrame({
    "dev_out_of_fold": {k: dev_oof[k] for k in ["precision", "recall", "f1", "flag_rate", "brier"]},
    "test": {k: test_metrics[k] for k in ["precision", "recall", "f1", "flag_rate", "brier"]},
})
gap["difference"] = gap["test"] - gap["dev_out_of_fold"]
gap.round(4)

### 3.5 Confusion matrix at the operating threshold
Counts and row percentages, so a manager can sanity-check every rate against real
bookings.

In [ ]:
cm = np.array([[test_metrics["tn"], test_metrics["fp"]], [test_metrics["fn"], test_metrics["tp"]]])
row_pct = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5.8, 4.8))
ax.imshow(row_pct, cmap=SEQUENTIAL_CMAP, vmin=0, vmax=1)
labels = [["True negative", "False positive"], ["False negative", "True positive"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{labels[i][j]}\n{cm[i, j]:,}\n({row_pct[i, j]:.0%} of row)", ha="center", va="center",
                color="white" if row_pct[i, j] > 0.55 else COLORS["ink"], fontsize=10)
ax.set_xticks([0, 1], ["Predicted: stays", "Predicted: cancels"])
ax.set_yticks([0, 1], ["Actual: stayed", "Actual: cancelled"])
ax.grid(False)
ax.set_title(f"Test confusion matrix (threshold {THRESHOLD:.2f})", loc="left")
save_figure(fig, "test_confusion_matrix")
plt.show()

### 3.6 Risk tiers on Test
Each tier's observed cancellation rate on unseen data is the clearest evidence for
stakeholders that the tiers mean something.

In [ ]:
tier = np.where(p_test >= THRESHOLD, "High", np.where(p_test >= MEDIUM, "Medium", "Low"))
tiers = (pd.DataFrame({"tier": tier, "y": y_test}).groupby("tier")
         .agg(bookings=("y", "size"), cancellations=("y", "sum")).reindex(["Low", "Medium", "High"]))
tiers["share_of_bookings"] = tiers["bookings"] / len(y_test)
tiers["cancellation_rate"] = tiers["cancellations"] / tiers["bookings"]
tiers["share_of_all_cancellations"] = tiers["cancellations"] / y_test.sum()
tiers.to_csv(PATHS["tables"] / "14_risk_tiers_test.csv")
tiers.round(3)

### 3.7 Business view: expected cost vs no model
Uses the cost assumptions locked in the policy (stated, not measured).

In [ ]:
costs = policy["cost_assumptions"]["per_outcome_costs"]
model_cost = (test_metrics["tp"] * costs["tp"] + test_metrics["fp"] * costs["fp"] + test_metrics["fn"] * costs["fn"]) / len(y_test) * 1000
no_model_cost = y_test.sum() * costs["fn"] / len(y_test) * 1000
print(f"Expected cost per 1,000 bookings: with model {model_cost:,.0f} vs no model {no_model_cost:,.0f} "
      f"({1 - model_cost / no_model_cost:.1%} lower, under the stated assumptions)")

## 4. Persist final results and write the lock

In [ ]:
final_table = ci_table.copy()
final_table["model_id"] = policy["model_id"]
final_table["threshold"] = THRESHOLD
final_table.to_csv(PATHS["tables"] / "final_test_metrics.csv")

if SCORE_TEST and not ALREADY_LOCKED:
    save_json({
        "evaluated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "policy_sha256": policy["policy_sha256"],
        "model_id": policy["model_id"],
        "test_features_sha256": test_hash,
        "headline_metrics": {k: float(test_metrics[k]) for k in ["pr_auc", "roc_auc", "precision", "recall", "f1", "brier", "accuracy"]},
        "expected_cost_per_1000": {"model": model_cost, "no_model": no_model_cost},
        "library_versions": LIBRARY_VERSIONS,
    }, LOCK_PATH)
    print("Test evaluation locked -> artifacts/test_evaluation_lock.json")

## 5. Appendix: Temporal robustness check (separate experiment)
### 5.1 Why
The primary split is random, but deployment predicts **future** bookings, and the
cancellation rate drifts upward over time (notebook 03). We refit the *same*
configuration (same hyper-parameters, same calibration method, same threshold rule) on
2015–16 arrivals and evaluate on 2017 arrivals. This experiment uses rows from all three
splits, so it is run only **after** the lock and is never used to change the policy.

In [ ]:
def temporal_pipeline(base_pipeline, model_card: dict):
    """Unfitted copy of the selected configuration; XGBoost's Dev early stopping is replaced by its found round count."""
    pipeline = clone(base_pipeline)
    model_params = pipeline.named_steps["model"].get_params()
    if model_params.get("early_stopping_rounds"):
        pipeline.set_params(model__early_stopping_rounds=None, model__n_estimators=model_card["best_iteration"] + 1)
    return pipeline


if RUN_TEMPORAL_CHECK:
    all_features = pd.concat([read_frame(PATHS["features"] / f"{name}.csv") for name in ("train", "dev", "test")], ignore_index=True)
    past = all_features[all_features["arrival_date_year"].isin(TEMPORAL_TRAIN_YEARS)]
    future = all_features[all_features["arrival_date_year"].isin(TEMPORAL_TEST_YEARS)]
    print(f"Temporal train {len(past):,} rows (prevalence {past[TARGET].mean():.3f}) | "
          f"temporal test {len(future):,} rows (prevalence {future[TARGET].mean():.3f})")

### 5.2 Refit on the past, evaluate on the future

In [ ]:
if RUN_TEMPORAL_CHECK:
    card = load_json(PATHS["model_outputs"] / policy["model_id"] / "model_card.json")
    temporal_model = temporal_pipeline(bundle["model"], card)
    temporal_model.fit(past.drop(columns=[TARGET, ID_COLUMN]), past[TARGET].astype(int))
    p_future_raw = temporal_model.predict_proba(future.drop(columns=[TARGET, ID_COLUMN]))[:, 1]
    # Re-use the locked calibrator only as a monotonic map; ranking metrics are unaffected by it.
    p_future = apply_calibration(bundle["calibration_method"], bundle["calibrator"], p_future_raw)
    temporal_metrics = evaluate_predictions(future[TARGET].astype(int), p_future, THRESHOLD)

    comparison = pd.DataFrame({"random_split_test": test_metrics, "temporal_2017": temporal_metrics}).loc[
        ["pr_auc", "roc_auc", "precision", "recall", "f1", "brier", "flag_rate"]]
    comparison["difference"] = comparison["temporal_2017"] - comparison["random_split_test"]
    comparison.to_csv(PATHS["tables"] / "14_temporal_robustness.csv")
    display(comparison.astype(float).round(4))

### 5.3 Visual comparison
Grouped bars on one shared 0–1 axis: random-split Test vs temporal 2017. Ranking metrics
are the fair comparison. Threshold metrics also reflect the prevalence shift.

In [ ]:
if RUN_TEMPORAL_CHECK:
    metrics_to_plot = ["pr_auc", "roc_auc", "precision", "recall", "f1"]
    x = np.arange(len(metrics_to_plot))
    fig, ax = plt.subplots(figsize=(9, 4))
    for offset, color, column, label in [(-0.18, SERIES_COLORS[0], "random_split_test", "Random split (Test)"),
                                         (0.18, SERIES_COLORS[1], "temporal_2017", "Train 2015-16 -> 2017")]:
        bars = ax.bar(x + offset, comparison.loc[metrics_to_plot, column].astype(float), width=0.34, color=color, label=label)
        ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=8, color=COLORS["ink_muted"])
    ax.set_xticks(x, metrics_to_plot)
    ax.set_ylim(0, 1.1)
    ax.set_title("Does performance hold when predicting the future?", loc="left")
    ax.legend(loc="upper right")
    ax.grid(axis="x", visible=False)
    save_figure(fig, "temporal_robustness")
    plt.show()

## 6. Reporting checklist
- Quote Test metrics **with** their 95 % CIs (§3.3) and the Dev→Test gap (§3.4).
- Report the tier table (§3.6) and the cost comparison (§3.7) with its assumptions stated.
- Report the temporal check (§5) as a limitation and robustness signal, not as a headline.
- Fill `docs/06-evaluation-plan.md` §6.4 and `docs/07-recommendation-limitations.md` from these
  tables. Never re-run Test with a changed policy.